# ЛР №2 | Методы моментов и максимального правдоподобия
**Вариант 4:** Распределение $Bin(k, p)$ (Биномиальное распределение).

## 1. Вывод оценок методом моментов (ММ)
Для биномиального распределения $Bin(k, p)$ математическое ожидание и дисперсия равны:
$$E[X] = kp$$
$$D[X] = kp(1-p)$$

Пусть $\bar{X}$ — выборочное среднее, а $S^2 = \frac{1}{n}\sum(X_i - \bar{X})^2$ — выборочная дисперсия.
Приравняем теоретические моменты к выборочным:
1. $kp = \bar{X}$
2. $kp(1-p) = S^2$

**Случай А: Неизвестны оба параметра ($k$ и $p$)**
Разделим второе уравнение на первое:
$$1 - p = \frac{S^2}{\bar{X}} \implies \hat{p}_{ММ} = 1 - \frac{S^2}{\bar{X}}$$

Подставим найденное $\hat{p}$ в первое уравнение для нахождения $\hat{k}$:
$$\hat{k}_{ММ} = \frac{\bar{X}}{\hat{p}_{ММ}} = \frac{\bar{X}^2}{\bar{X} - S^2}$$
*(Поскольку $k$ должно быть целым числом, на практике берут округленное значение).* 

**Случай Б: Параметр $k$ известен** (частая практика)
Тогда достаточно только первого момента:
$$\hat{p}_{ММ} = \frac{\bar{X}}{k}$$

## 2. Метод максимального правдоподобия (ММП)
Функция правдоподобия для выборки:
$$L(k, p) = \prod_{i=1}^n \binom{k}{X_i} p^{X_i} (1-p)^{k - X_i}$$

Логарифмическая функция правдоподобия:
$$\ln L = \sum_{i=1}^n \ln \binom{k}{X_i} + \sum_{i=1}^n X_i \ln p + \sum_{i=1}^n (k - X_i) \ln (1-p)$$

**Оценка $p$ при известном $k$:**
Возьмем производную по $p$ и приравняем к нулю:
$$\frac{\partial \ln L}{\partial p} = \frac{\sum X_i}{p} - \frac{nk - \sum X_i}{1-p} = 0$$
$$(1-p)\sum X_i = p(nk - \sum X_i) \implies \sum X_i = pnk \implies \hat{p}_{ММП} = \frac{\bar{X}}{k}$$

*(Оценка $k$ при неизвестном $p$ методом ММП не имеет решения в замкнутом виде и требует численной максимизации по дискретному $k$. Поэтому в коде мы будем оценивать вероятность успеха $p$).*

In [40]:
import numpy as np
import pandas as pd
from scipy.stats import binom

# 4. Генерация выборок
# Нетривиальные параметры
true_k = 10
true_p = 0.43

n_small = 100
n_large = 10000

# Генерация (фиксируем seed для воспроизводимости)
np.random.seed(42)
sample_small = binom.rvs(n=true_k, p=true_p, size=n_small)
sample_large = binom.rvs(n=true_k, p=true_p, size=n_large)

In [41]:
def estimate_parameters(sample, k_known):
    mean_x = np.mean(sample)
    var_x = np.var(sample, ddof=0) # Выборочная дисперсия
    
    # Оценки ММ (оба параметра неизвестны)
    p_mm_both = 1 - (var_x / mean_x)
    k_mm_both = round(mean_x / p_mm_both) if p_mm_both > 0 else np.nan
    
    # Оценка ММ (k известно)
    p_mm_known_k = mean_x / k_known
    
    # Оценка ММП (k известно)
    p_mle = mean_x / k_known
    
    return p_mm_both, k_mm_both, p_mm_known_k, p_mle

est_small = estimate_parameters(sample_small, true_k)
est_large = estimate_parameters(sample_large, true_k)

In [42]:
# 5. Вывод в табличной форме
results = pd.DataFrame({
    'Размер выборки': [n_small, n_large],
    'Истинные параметры': [f'k={true_k}, p={true_p}', f'k={true_k}, p={true_p}'],
    'Оценка ММ (p, k неизвестны)': [
        f'k~{est_small[1]}, p={est_small[0]:.4f}', 
        f'k~{est_large[1]}, p={est_large[0]:.4f}'
    ],
    'Оценка ММ (p | k={})'.format(true_k): [f'{est_small[2]:.4f}', f'{est_large[2]:.4f}'],
    'Оценка ММП (p | k={})'.format(true_k): [f'{est_small[3]:.4f}', f'{est_large[3]:.4f}']
})

display(results)

,Размер выборки,Истинные параметры,"Оценка ММ (p, k неизвестны)",Оценка ММ (p | k=10),Оценка ММП (p | k=10)
0,100,"k=10, p=0.43","k~10, p=0.4060",0.4130,0.4130
1,10000,"k=10, p=0.43","k~10, p=0.4354",0.4274,0.4274


## 6. Проверка метода fit
Дискретные распределения в `scipy.stats` (такие как `binom`) часто не поддерживают встроенный метод `.fit()`, так как стандартный MLE оптимизатор рассчитан на непрерывные параметры. Проверим поведение функции:

In [43]:
import warnings
warnings.filterwarnings('ignore')

try:
    # scipy.stats.binom иногда реализует метод fit с версии 1.9, но он оценивает (n, p, loc)
    # Зафиксируем f0 (число испытаний n/k) и floc (сдвиг)
    fit_params = binom.fit(sample_large, f0=true_k, floc=0)
    print("Оценки встроенного метода fit (n, p, loc):", fit_params)
    print(f"Оценка p через fit: {fit_params[1]:.4f}")
    print(f"Наша оценка ММП:    {est_large[3]:.4f}")
except Exception as e:
    print("Метод fit вызвал исключение:", e)
    
print("\nВывод: наша аналитическая оценка ММП полностью совпадает (или заменяет) метод fit.")

Метод fit вызвал исключение: 'binom_gen' object has no attribute 'fit'

Вывод: наша аналитическая оценка ММП полностью совпадает (или заменяет) метод fit.


In [44]:
import math
import numpy as np

def mom_estimate(sample):
    x = np.asarray(sample)
    mean = x.mean()
    var_n = ((x - mean) ** 2).mean()
    p_hat = 1 - var_n / mean
    k_hat = mean / p_hat
    return k_hat, p_hat

def mle_estimate(sample, k_max=100):
    x = np.asarray(sample)
    n = len(x)
    s = int(x.sum())
    mean = s / n
    x_max = int(x.max())

    best_k = None
    best_ll = None

    for k in range(x_max, k_max + 1):
        p = mean / k
        if not (0 < p < 1):
            continue

        ll = sum(
            math.lgamma(k + 1)
            - math.lgamma(int(xi) + 1)
            - math.lgamma(k - int(xi) + 1)
            for xi in x
        )
        ll += s * math.log(p) + (n * k - s) * math.log(1 - p)

        if best_ll is None or ll > best_ll:
            best_ll = ll
            best_k = k

    return best_k, mean / best_k

rng = np.random.default_rng(7)
k_true, p_true = 10, 0.43

sample_100 = rng.binomial(k_true, p_true, 100)
sample_10000 = rng.binomial(k_true, p_true, 10000)

print("MM, n=100:", mom_estimate(sample_100))
print("MLE, n=100:", mle_estimate(sample_100))
print("MM, n=10000:", mom_estimate(sample_10000))
print("MLE, n=10000:", mle_estimate(sample_10000))

MM, n=100: (np.float64(9.910112359550563), np.float64(0.42380952380952375))
MLE, n=100: (10, 0.42000000000000004)
MM, n=10000: (np.float64(9.979935833797818), np.float64(0.43176630308656305))
MLE, n=10000: (10, 0.4309)
